In [ ]:
!pip install --upgrade google-cloud-aiplatform[evaluation]

In [1]:
#init required libraries
import pandas as pd
import json
import vertexai
from io import StringIO
from vertexai.generative_models import Part, Tool, grounding
from vertexai.evaluation import EvalTask, PointwiseMetric, PairwiseMetric
import vertexai.evaluation.metrics.pointwise_metric as pointwise_metric

pd.options.display.max_colwidth = 300

vertexai.init(location="us-central1")
def analyze_gemini(contents, model_name, instruction, response_mime, response_schema, token_limit, bUse_Grounding):
    from vertexai.generative_models import GenerationConfig, GenerativeModel, HarmCategory, HarmBlockThreshold
    def get_model():
        return GenerativeModel(model_name)
        #return GenerativeModel(model_name, system_instruction=instruction)
    generation_config=GenerationConfig(
        candidate_count = 1,
        max_output_tokens = token_limit,
        temperature = 0,
        top_p = 0.5,
        top_k = 1,
        response_mime_type = response_mime,
        response_schema = response_schema
    )

    tool_google_search = Tool.from_google_search_retrieval(grounding.GoogleSearchRetrieval())

    responses = get_model().generate_content(
        contents=contents,
        generation_config=generation_config,
        safety_settings={
            HarmCategory.HARM_CATEGORY_UNSPECIFIED: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        },
        stream=False, 
        tools=[tool_google_search] if bUse_Grounding else None
    )

    return responses.text

* LLM 을 이용해 번역 엔진을 만들고자 합니다. Human prefrence 가 있는 Reference 가 존재하는 경우 번역 결과에 대해 번역 품질을 LLM 이 평가할 수 있습니다.
* 이 평가 결과를 통해 LLM 엔진의 Prompt 를 개선하는 작업을 수행하면 됩니다.
* 개선된 Prompt 로 새로운 컨텐츠를 번역하고, 이 번역결과는 reference 가 없기 때문에 reference free 모드로 번역 품질을 살펴봅니다.

In [2]:
# Pointwise 검사를 위한 데이터셋 준비
# Pointwise는 단일 모델의 결과에 대한 평가입니다.
data = [
    ["This account is not protected with a two-factor authentication.", 
     "2차 인증이 되지 않은 계정입니다.",
     "이중 인증으로 보호되지 않은 계정입니다."],
    ["Even the longest journey begins with a single step.",
    "천 리 길도 한 걸음부터.",
    "가장 먼 여정도 한 걸음에서 시작됩니다."],
    ["Even the longest journey begins with a single step.",
    "Лиха беда начало.",
    "Даже самая длинная дорога продолжается с одного шага."]
]
pointwise_df = pd.DataFrame(data, columns=['prompt', 'reference', 'response'])
pointwise_df

,prompt,reference,response
0,This account is not protected with a two-factor authentication.,2차 인증이 되지 않은 계정입니다.,이중 인증으로 보호되지 않은 계정입니다.
1,Even the longest journey begins with a single step.,천 리 길도 한 걸음부터.,가장 먼 여정도 한 걸음에서 시작됩니다.
2,Even the longest journey begins with a single step.,Лиха беда начало.,Даже самая длинная дорога продолжается с одного шага.


In [3]:
from vertexai.evaluation import EvalTask, PointwiseMetric, PairwiseMetric
pointwise_ground_truth_metric_prompt = """
# Instruction
당신은 전문적인 번역 품질 평가자 입니다. 당신은 AI 모델이 번역한 내용의 품질을 평가해야 합니다.
우리는 당신에게 'Source, AI translation, Ground Truth'를 제공할 것입니다.
업무를 수행하기 위해 'Source, AI translation, Ground Truth'을 신중하게 읽고, 아래의 'Evaluation' 섹션에 정의된 'Evaluation criteria'에 근거하여 평가해야 합니다.
당신은 'Rating rubric'과 'Evaluation steps'에 기반하여 평가해야 합니다. 평가의 이유에 대해 단계별로 '한글'로 설명하고 'Rating rubric'의 순위만 선택해야 합니다.

# Evaluation
## Metric Definition
번역이 'Ground Truth'와 동일하게 번역되었는지, 그렇지 않으면 의미적으로 유사하고 간결, 명확, 캐주얼하게 작성되었는지 전반적으로 확인합니다.

## Evaluation criteria
예시 일치성: 예시로 제공되는 'Ground Truth'와 얼마나 일치하는지 확인합니다.
지시를 잘 따르는지: 'AI translation'이 'Evaluation Definition'을 잘 따라서 수행됐는지 확인합니다.

## Rating rubric
5: (매우 좋음). 'Ground Truth'와 글자수 등 표현이 정확하게 일치함
4: (좋음). 'Ground Truth'와 어순 외 차이 없음
3: (괜찮음). 속담이나 관용적 표현을 사용하지 못했지만 이해하는데 큰 문제 없음
2: (나쁨). 이해하기가 어려움
1: (매우 나쁨). 번역결과가 'Ground Truth'와 매우 다름

## Evaluation steps
STEP 1: 'Ground Truth'을 참고하여 'AI translation'이 번역한 내용을 비교합니다.
STEP 2: 평가 규칙에 따라 점수를 부여합니다.

# Source, AI translation, Ground Truth
### Source
{prompt}

### Ground Truth
{reference}

### AI translation
{response}
"""

pointwise_ground_truth_text_quality = PointwiseMetric(
    metric="pointwise_ground_truth_quality",
    metric_prompt_template=pointwise_ground_truth_metric_prompt,
)
#For MetricX, require source field
pointwise_df['source'] = pointwise_df['prompt']
eval_task = EvalTask(dataset=pointwise_df, 
                    metrics=[pointwise_ground_truth_text_quality, "bleu", "rouge", pointwise_metric.MetricX(), pointwise_metric.Comet()],
                    experiment="pointwise-eval")
result_pointwise = eval_task.evaluate()

#comet, higher is better (0-1), reference based regression approach
#metricx, lower is better (0-25), reference is optional
result_pointwise.metrics_table

Associating projects/1045259343465/locations/us-central1/metadataStores/default/contexts/pointwise-eval-9cb97731-b838-4ec2-805e-38541d0f7e0d to Experiment: pointwise-eval


Computing metrics with a total of 15 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 15/15 [00:08<00:00,  1.75it/s]

All 15 metric requests are successfully computed.
Evaluation Took:8.576858434942551 seconds


,prompt,reference,response,source,pointwise_ground_truth_quality/explanation,pointwise_ground_truth_quality/score,bleu/score,rouge/score,metricx/score,comet/score
0,This account is not protected with a two-factor authentication.,2차 인증이 되지 않은 계정입니다.,이중 인증으로 보호되지 않은 계정입니다.,This account is not protected with a two-factor authentication.,"'이중 인증'과 '2차 인증'은 의미적으로 유사하며, 어순 외 다른 차이는 없어 '좋음'에 해당합니다.",4.0,0.302138,0.0,0.961893,0.915181
1,Even the longest journey begins with a single step.,천 리 길도 한 걸음부터.,가장 먼 여정도 한 걸음에서 시작됩니다.,Even the longest journey begins with a single step.,"AI 번역이 '천 리 길도 한 걸음부터'라는 속담을 사용하지 못했지만, 의미는 유사하여 이해하는 데 큰 문제는 없으므로 3점을 부여합니다.",3.0,0.078099,0.0,0.964249,0.925086
2,Even the longest journey begins with a single step.,Лиха беда начало.,Даже самая длинная дорога продолжается с одного шага.,Even the longest journey begins with a single step.,"AI translation은 주어진 source text를 직역하여 '가장 긴 여정조차 한 걸음부터 시작한다'는 의미를 전달하지만, Ground Truth는 '시작이 반이다'라는 러시아 속담으로, 의미가 매우 다르므로 1점을 부여합니다.",1.0,0.047677,0.0,5.247432,0.568027


In [4]:
# Pairwise 검사를 위한 데이터셋 준비
# Pairwise 는 2개 모델의 성능을 비교하기 위한 방법입니다.
#prompt(Source), reference, response (Response B), baseline_model_response (Response A)
data = [
    ["This account is not protected with a two-factor authentication.", 
     "2차 인증이 되지 않은 계정입니다.",
     "이중 인증으로 보호되지 않은 계정입니다.",
     "이 계정은 이중 인증으로 안전하게 보호되지 않습니다."],
    ["Even the longest journey begins with a single step.",
    "천 리 길도 한 걸음부터.",
    "가장 먼 여정도 한 걸음에서 시작됩니다.",
    "가장 긴 여행조차도 한 걸음부터 시작됩니다."],
    ["Even the longest journey begins with a single step.",
    "Лиха беда начало.",
    "Даже самая длинная дорога продолжается с одного шага.",
    "Даже самый долгий путь начинается с первого шага."]
]
pairwise_df = pd.DataFrame(data, columns=['prompt', 'reference', 'response', 'baseline_model_response'])
pairwise_df

,prompt,reference,response,baseline_model_response
0,This account is not protected with a two-factor authentication.,2차 인증이 되지 않은 계정입니다.,이중 인증으로 보호되지 않은 계정입니다.,이 계정은 이중 인증으로 안전하게 보호되지 않습니다.
1,Even the longest journey begins with a single step.,천 리 길도 한 걸음부터.,가장 먼 여정도 한 걸음에서 시작됩니다.,가장 긴 여행조차도 한 걸음부터 시작됩니다.
2,Even the longest journey begins with a single step.,Лиха беда начало.,Даже самая длинная дорога продолжается с одного шага.,Даже самый долгий путь начинается с первого шага.


In [5]:
pairwise_model_compare_metric_prompt = """
# Instruction
당신은 전문적인 번역 품질 평가자 입니다. 당신은 2개의 AI 모델이 번역한 내용의 품질을 평가해야 합니다.
우리는 당신에게 'Source, Response A, Response B, Ground Truth'를 제공할 것입니다.
업무를 수행하기 위해 'Source, Response A, Response B, Ground Truth'을 신중하게 읽고, 아래의 '평가' 섹션에 정의된 '평가항목'에 근거하여 평가해야 합니다.
당신은 'Evaluation rule'과 'Evaluation steps'에 기반하여 평가해야 합니다. 평가의 이유에 대해 단계별로 '한글'로 설명하고 'Evaluation rule'의 순위만 선택해야 합니다.

# Evaluation
## Metric Definition
번역이 'Ground Truth'와 동일하게 번역되었는지, 그렇지 않으면 의미적으로 유사하고 간결, 명확, 캐주얼하게 작성되었는지 전반적으로 확인합니다.

## Evaluation criteria
예시 일치성: 예시로 제공되는 'Ground Truth'와 얼마나 일치하는지 확인합니다.
지시를 잘 따르는지: 'AI translation'이 'Evaluation Definition'을 잘 따라서 수행됐는지 확인합니다.

## Rating rubric
STEP 1: Analyze Response A based on all the Criteria.
STEP 2: Analyze Response B based on all the Criteria.
STEP 3: Compare the overall performance of Response A and Response B based on your analyses and assessment.
STEP 4: Output your preference of "A", "SAME" or "B" to the pairwise_choice field according to the Rating Rubric.
STEP 5: Output your assessment reasoning in the explanation field.

# Source, Response A, Response B, Ground Truth
### Source
{prompt}

### Ground Truth
{reference}

### Response A
{baseline_model_response}

### Response B
{response}
"""

pairwise_ground_truth_text_quality = PairwiseMetric(
    metric="pairwise_ground_truth_quality",
    metric_prompt_template=pairwise_model_compare_metric_prompt,
)

eval_task = EvalTask(dataset=pairwise_df, 
                    metrics=[pairwise_ground_truth_text_quality],
                    experiment="pairwise-eval")
result_pairwise = eval_task.evaluate()
result_pairwise.metrics_table.rename(columns={"baseline_model_response": "Response A", "response": "Response B"}).replace(['BASELINE', 'CANDIDATE'], ['A', 'B'])

Associating projects/1045259343465/locations/us-central1/metadataStores/default/contexts/pairwise-eval-38a49fe4-1b54-4a24-a1b5-c5895fa4b462 to Experiment: pairwise-eval


Computing metrics with a total of 3 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 3/3 [00:02<00:00,  1.33it/s]

All 3 metric requests are successfully computed.
Evaluation Took:2.2666387179633602 seconds


,prompt,reference,Response B,Response A,pairwise_ground_truth_quality/explanation,pairwise_ground_truth_quality/pairwise_choice
0,This account is not protected with a two-factor authentication.,2차 인증이 되지 않은 계정입니다.,이중 인증으로 보호되지 않은 계정입니다.,이 계정은 이중 인증으로 안전하게 보호되지 않습니다.,"두 응답 모두 '2차 인증이 되지 않은 계정입니다'라는 Ground Truth와 유사하게 번역되었으며, 의미적으로도 차이가 없고 간결하고 명확하게 작성되었습니다.",TIE
1,Even the longest journey begins with a single step.,천 리 길도 한 걸음부터.,가장 먼 여정도 한 걸음에서 시작됩니다.,가장 긴 여행조차도 한 걸음부터 시작됩니다.,"Response A와 CANDIDATE response 모두 '가장 먼 여정도 한 걸음에서 시작됩니다'와 유사한 의미를 전달하며, 문법적으로도 정확하고 자연스럽습니다. 두 응답 간에 뚜렷한 품질 차이가 없으므로 동일하다고 평가합니다.",TIE
2,Even the longest journey begins with a single step.,Лиха беда начало.,Даже самая длинная дорога продолжается с одного шага.,Даже самый долгий путь начинается с первого шага.,"BASELINE response is more concise and accurate, adhering closely to the meaning of the source text, while CANDIDATE response has a minor error in translation.",A


* Reference 가 없는, 새로운 컨텐츠에 대한 번역과 이를 MetricX를 이용해 reference free (Quality Estimation, QE) 모드로 평가하는 예시입니다.

In [6]:
prompt = """당신을 비디오를 분석해서 transcript를 작성해야 하는 AI Assistant 입니다.
아래 가이드라인에 맞게 transcription을 작성해주세요.

1. 첨부된 비디오를 분석하여 아래와 같은 포맷으로 모든 대화 내용을 빠짐없이 출력해주세요.
2. 결과 출력 단위는 비디오 내의 장면이 구분되는 특정 장소를 기준으로 나누어서 출력해주세요.
3. 목소리를 기반으로 화자(speaker)를 정확하게 분리해서 영어로 출력해주세요.
4. 목소리외에 다양한 효과음, 감정표현은 괄호를 사용해서 반드시 자세히 표현해주세요."""
response_schema = {
    "type": "ARRAY",
    "items": {
        "type": "OBJECT",
        "properties": {
            "location": { "type": "STRING",},
            "start_time": { "type": "STRING",},
            "end_time": { "type": "STRING",},
            "elapsed_time": { "type": "STRING",},
            "transcription": {
                "type": "ARRAY",
                "items" : {
                  "type": "OBJECT",
                  "properties": {
                    "speaker": { "type": "STRING",},
                    "transcript": { "type": "STRING",},
                  }
                }
            },
        },
        "required": ["start_time","end_time","elapsed_time"],
    },
}
video = Part.from_uri(mime_type="video/*", uri="https://www.youtube.com/watch?v=OoUVSHDbAeM")
content = [
    prompt,
    video
]
response = analyze_gemini(content, "gemini-2.5-pro", "", "application/json", response_schema, 65536, False)
pd_transcription = pd.json_normalize(json.loads(response), record_path='transcription', meta=['location', 'start_time', 'end_time', 'elapsed_time'])
pd_transcription

,speaker,transcript,location,start_time,end_time,elapsed_time
0,Announcer,(Train station announcement) ...trains for London King's Cross.,Train Station,00:00:00,00:00:26,00:00:26
1,Friend,"Come on, Stephen!",Train Station,00:00:00,00:00:26,00:00:26
2,Friend,Get a move on!,Train Station,00:00:00,00:00:26,00:00:26
3,Friend,"What's wrong with you, man?",Train Station,00:00:00,00:00:26,00:00:26
4,Friend,Chop-chop!,Train Station,00:00:00,00:00:26,00:00:26
5,Sound,(Train whistle blows and steam hisses),Train Station,00:00:00,00:00:26,00:00:26
6,Professor,"A star... more than three times the size of our sun, ought to end its life, how?",Lecture Hall,00:00:26,01:27:00,00:01:01
7,Professor,With a collapse.,Lecture Hall,00:00:26,01:27:00,00:01:01
8,Professor,The gravitational forces of the entire mass overcoming the electromagnetic forces of individual atoms and so collapsing inwards.,Lecture Hall,00:00:26,01:27:00,00:01:01
9,Professor,"If the star is massive enough, it will continue this collapse, creating a black hole where the warping of spacetime is so great that nothing can escape, not even light.",Lecture Hall,00:00:26,01:27:00,00:01:01


In [7]:
source_text = pd_transcription[['location', 'speaker', 'transcript']].to_csv(index=False)
content = [
    video,
    """
    다음은 위 영상에 대해 CSV 로 만들어진 영화 대사입니다. CSV 헤더에는 'location', 'speaker', 'transcript' 로 구성돼 있습니다.
    'transcript'의 내용을 영상의 내용을 참고하여 번역하여 'translated' 에 결과를 알려주세요.
    예시의 CSV 포맷으로 헤더를 포함하여 출력해 주세요.
    Remove any leading/trailing lines.
    Always wrap values using quotes.
    example:
    location,speaker,transcript,translated
    "Restraunt","Mike","Hello, Good Day!","안녕하세요, 좋은 하루입니다!"
    """,
    source_text
]
output = analyze_gemini(content, "gemini-2.5-pro", "", "text/plain", None, 65536, False)
result = pd.read_csv(StringIO(output))
result

,location,speaker,transcript,translated
0,Train Station,Announcer,(Train station announcement) ...trains for London King's Cross.,(기차역 안내 방송) ...런던 킹스크로스행 열차입니다.
1,Train Station,Friend,"Come on, Stephen!","스티븐, 어서!"
2,Train Station,Friend,Get a move on!,서둘러!
3,Train Station,Friend,"What's wrong with you, man?","대체 왜 그래, 친구?"
4,Train Station,Friend,Chop-chop!,빨리빨리!
5,Train Station,Sound,(Train whistle blows and steam hisses),(기차 경적 소리와 증기 소리)
6,Lecture Hall,Professor,"A star... more than three times the size of our sun, ought to end its life, how?","별... 우리 태양보다 3배 이상 큰 별은, 어떻게 생을 마감해야 할까요?"
7,Lecture Hall,Professor,With a collapse.,붕괴로요.
8,Lecture Hall,Professor,The gravitational forces of the entire mass overcoming the electromagnetic forces of individual atoms and so collapsing inwards.,전체 질량의 중력이 개별 원자의 전자기력을 이겨내고 안으로 붕괴하는 거죠.
9,Lecture Hall,Professor,"If the star is massive enough, it will continue this collapse, creating a black hole where the warping of spacetime is so great that nothing can escape, not even light.","별의 질량이 충분히 크다면, 이 붕괴는 계속되어 시공간의 왜곡이 너무 커서 빛조차 빠져나올 수 없는 블랙홀을 만들게 됩니다."


In [8]:
eval_task = EvalTask(dataset=result[['transcript', 'translated']].rename(columns={'transcript': 'source', 'translated': 'response'}),
                    metrics=[pointwise_metric.MetricX(version="METRICX_24_SRC")],
                    experiment="pointwise-eval")
result_pointwise = eval_task.evaluate()
result_pointwise.metrics_table

#metricx, lower is better (0-25)
#From the paper, https://aclanthology.org/2024.wmt-1.35/
# 0 ~ 1 is good
# above 5 means undertranslation

Associating projects/1045259343465/locations/us-central1/metadataStores/default/contexts/pointwise-eval-98bb493a-f0d8-4e35-b03a-627f6994b3b8 to Experiment: pointwise-eval


Computing metrics with a total of 42 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 42/42 [00:12<00:00,  3.29it/s]

All 42 metric requests are successfully computed.
Evaluation Took:12.762580803013407 seconds


,source,response,metricx/score
0,(Train station announcement) ...trains for London King's Cross.,(기차역 안내 방송) ...런던 킹스크로스행 열차입니다.,2.563894
1,"Come on, Stephen!","스티븐, 어서!",3.517939
2,Get a move on!,서둘러!,2.659924
3,"What's wrong with you, man?","대체 왜 그래, 친구?",2.261599
4,Chop-chop!,빨리빨리!,5.132485
5,(Train whistle blows and steam hisses),(기차 경적 소리와 증기 소리),4.179062
6,"A star... more than three times the size of our sun, ought to end its life, how?","별... 우리 태양보다 3배 이상 큰 별은, 어떻게 생을 마감해야 할까요?",4.623527
7,With a collapse.,붕괴로요.,4.691329
8,The gravitational forces of the entire mass overcoming the electromagnetic forces of individual atoms and so collapsing inwards.,전체 질량의 중력이 개별 원자의 전자기력을 이겨내고 안으로 붕괴하는 거죠.,2.587744
9,"If the star is massive enough, it will continue this collapse, creating a black hole where the warping of spacetime is so great that nothing can escape, not even light.","별의 질량이 충분히 크다면, 이 붕괴는 계속되어 시공간의 왜곡이 너무 커서 빛조차 빠져나올 수 없는 블랙홀을 만들게 됩니다.",1.824070
